# 0 - Helpers

In [ ]:
from generate_frontier import *
from matplotlib.patches import Rectangle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import country_converter as coco
import pypsa
import os

def plot_hourly_energy_matrix(df, style_map=None, use_pointers=True, label_offset=2, title="Global Market"):
    """
    Plots each column of the DataFrame with its own style and inline labels.

    Parameters:
    - df: pandas DataFrame with columns to plot.
    - style_map: dict mapping column names to style kwargs.
    - use_pointers: bool, whether to draw arrows from text to line ends.
    - label_offset: int/float, how far to place the label from the line end (in x-units).
    """
    
    fig, ax = plt.subplots(figsize=(8, 8))

    for col in df.columns:
        style = style_map.get(col, {}) if style_map else {}
        line, = ax.plot(df.index, df[col], label=col, **style)

        # plot a vertical line with the x being the last non Nan value, and the y starting from 0 to the last y value
        last_valid_index = df[col].last_valid_index()

        if last_valid_index > 100:
            x_end = 100
            y_end = df[col].loc[100]
        
        else:
            x_end = last_valid_index
            y_end = df[col].loc[last_valid_index]
    
            # Plot vertical line
            ax.plot([x_end, x_end], [0, y_end], **style)

        # Position for the label
        if x_end < 80:
            x_text = x_end + label_offset
            ha='left'
        else:
            x_text = x_end - label_offset
            ha='right'
        
        y_text = y_end

        if use_pointers:
            # Draw arrow from text to the line end
            ax.annotate(
                col,
                xy=(x_end, y_end),
                xytext=(x_text, y_text),
                textcoords='data',
                ha=ha,
                va='center',
                arrowprops=dict(arrowstyle="->", color=style.get('color', line.get_color())),
                fontsize=10,
                color=style.get('color', line.get_color())
            )

    ax.set_xlim([0, 100 + label_offset * 2])
    ax.set_ylim([40, 100]) # None is below 40%
    ax.set_title(title)
    ax.set_xlabel("Energy Matching")
    ax.set_ylabel("Hourly Matching")
    ax.grid()
    ax.set_box_aspect(1)
    
    fig.tight_layout()    

    return fig

countries = [
    'AL', 'AT', 'BA', 'BE', 'BG', 'CH', 'CZ', 'DE', 'DK',
    'EE', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE',
    'IT', 'LT', 'LU', 'LV', 'ME', 'MK', 'NL', 'NO',
    'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'XK'
]
cc = coco.CountryConverter()

country_colors = {
    'Albania': '#E41B17',
    'Austria': '#ED2939',
    'Belgium': '#FDDA24',
    'Bosnia and Herzegovina': '#0033A0',
    'Bulgaria': '#009900',
    'Switzerland': '#FF0000',
    'Czechia': '#11457E',
    'Germany': '#DD0000',
    'Denmark': '#C60C30',
    'Estonia': '#0072CE',
    'Spain': '#AA151B',
    'Finland': '#003580',
    'France': '#0055A4',
    'United Kingdom': '#00247D',
    'Greece': '#0D5EAF',
    'Croatia': '#FF0000',
    'Hungary': '#436F4D',
    'Ireland': '#009A49',
    'Italy': '#009246',
    'Lithuania': '#FDB913',
    'Luxembourg': '#00A3E0',
    'Latvia': '#8C1C13',
    'Montenegro': '#C8102E',
    'North Macedonia': '#E62020',
    'Netherlands': '#21468B',
    'Norway': '#BA0C2F',
    'Poland': '#DC143C',
    'Portugal': '#006600',
    'Romania': '#002B7F',
    'Serbia': '#C8102E',
    'Sweden': '#005B99',
    'Slovenia': '#005BBB',
    'Slovakia': '#0B4EA2',
    'Kosovo': '#0066CC',
    'EU': 'blue',
}

scenario_colors = {
    "baseline":"blue",
    "co2-price25":"black",
    "co2-price50":"red",
    "co2-price100":"orange",
    "rps":"green"
}

year_linestyle = {
    2025: 'solid',
    2030: 'dashdot',
    2035: 'dashed',
    2040: 'dotted',
}

## Comparing different years of the same profile
style_map_EU = {
    f"EU_{year}": {'color': country_colors["EU"], 'linewidth': 2, 'linestyle': linestyle}
    for year, linestyle in year_linestyle.items()
}


style_map_country_year = {
    f"{country}_{year}": {'color': color, 'linewidth': 2, 'linestyle': linestyle}
    for country, color in country_colors.items()
    for year, linestyle in year_linestyle.items()
}

style_map_baseline = {
    f"{sc}_{year}": {'color': colors, 'linewidth': 2, 'linestyle': linestyle}
    for year, linestyle in year_linestyle.items() 
    for sc, colors in scenario_colors.items()
}

# 1 - Retrieve networks

In [ ]:
# For all scenarios with 4 timesteps
years = [2025, 2030, 2035, 2040]
scenarios = [
    "baseline",
    "baseline-rps",
    "baseline-co2-price25",
    "baseline-co2-price50",
    "baseline-co2-price100",
    "energy-match-25",
    "hourly-match-25-90",
    "hourly-match-25-95",
    "hourly-match-25-98",
    "hourly-match-25-99",
    "hourly-match-EU-25-99",
    "hourly-match-no-LDES-25-99",
    "hourly-match-no-clean-firm-25-99",
    "hourly-match-co2-price25-25-99",
    "hourly-match-co2-price50-25-99",
    "hourly-match-co2-price100-25-99",
    "hourly-match-noadd-10-99",
    "hourly-match-noadd-50-99",
    "hourly-match-noadd-90-99",
]

# For all scenarios with 2 timesteps
# years = [2025, 2030]
# scenarios = ["energy-match-50",
#              "hourly-match-50-90",
#              "hourly-match-50-95",
#              "hourly-match-50-98",
#              "hourly-match-50-99",
#              "hourly-match-EU-50-99",
#              "hourly-match-no-LDES-50-99",
#              "hourly-match-no-clean-firm-50-99",
#              "hourly-match-co2-price25-50-99",
#              "hourly-match-co2-price50-50-99",
#              ]

df_all = pd.DataFrame(index=scenarios, columns=years)

# Fill it
for sc in df_all.index:
    for year in df_all.columns:
        try:
            n = pypsa.Network(f"../results/{sc}/networks/base_s_39___{year}.nc")
        except:
            print(f"{sc}-{year} not availabe")
            continue
        n.name = f"{sc}-{year}"
        df_all.loc[sc, year] = n


# 2 - Frontier plots

In [ ]:
score = "cfe"
save_fig = True
show_fig = False

## (a-0) Comparing baseline scenarios

In [ ]:
fig_path = "figures/energy_procurement_frontier/system"

df_all_b = df_all[df_all.index.str.contains("baseline")]
df_all_b.index = df_all_b.index.str.replace("baseline-","")

if not df_all_b.empty:
    data = {}
    for sc in df_all_b.index:    
        for year in df_all.columns:
            n = df_all_b.loc[sc, year]
            res, load = get_score_load(n, score=score)
            
            global_res = res.sum(axis=1)
            global_load = load.sum(axis=1)
            
            data[f"{sc}_{year}"] = get_hourly_energy_matrix(n, global_res, global_load)
    
    df_results = pd.DataFrame(data, index=range(1, 120))
    fig = plot_hourly_energy_matrix(
        df_results,
        style_map_baseline, 
        use_pointers=False,
        title=f"baseline sensitivity {score.upper()} Global Market Energy Procurement Frontier"
    )
    
    r = Rectangle((0,0), 1, 1, fill=False, edgecolor='none', visible=False)
    
    handles, labels = fig.axes[0].get_legend_handles_labels()
    
    handles = handles[0:20:4] + [r] + handles[4:8]
    labels = [l[:-5] for l in labels[0:20:4]] + [""] + [l[-4:] for l in labels[4:8]]
    
    fig.axes[0].legend(
        handles, labels, ncol=1, loc='upper left', bbox_to_anchor=[0.0,1]
    )
    
    title_fig = f"{score.upper()} Frontier - Sensitivity"
    
    if save_fig:
        os.makedirs(fig_path, exist_ok=True)
        fig.savefig(f"{fig_path}/{title_fig}.png", dpi=300, bbox_inches="tight")

## (a-1) Comparing different years of the same profile

In [ ]:
fig_path = "figures/energy_procurement_frontier/system"
for sc in df_all.index:
    data = {}
    
    for year in df_all.columns:
        n = df_all.loc[sc, year]
        res, load = get_score_load(n, score=score)
        
        global_res = res.sum(axis=1)
        global_load = load.sum(axis=1)
        
        data[f"EU_{year}"] = get_hourly_energy_matrix(n, global_res, global_load)

    df_results = pd.DataFrame(data, index=range(1, 120))
    fig = plot_hourly_energy_matrix(df_results, style_map_EU, title=f"{sc} {score.upper()} Global Market Energy Procurement Frontier")
    title_fig = f"{score.upper()} Frontier - {sc} - EU"

    if save_fig:
        eu_fig_path = os.path.join(fig_path, sc)
        os.makedirs(eu_fig_path, exist_ok=True)
        fig.savefig(os.path.join(eu_fig_path, f"{title_fig}.png"), dpi=300, bbox_inches="tight")

    if not show_fig:
        plt.close()

## (a-2) Comparing different years of the same countries

In [ ]:
fig_path = "figures/energy_procurement_frontier/country"
for country in countries:
    
    for sc in df_all.index:
        data = {}
        
        for year in df_all.columns:
            n = df_all.loc[sc, year]
            res, load = get_score_load(n, score=score)
            
            national_res = res[country]
            national_load = load[country]
    
            country_name = cc.convert(names=country, to='name_short')
            data[f"{country_name}_{year}"] = get_hourly_energy_matrix(n, national_res, national_load)

        df_results = pd.DataFrame(data, index=range(1, 120))
        fig = plot_hourly_energy_matrix(
            df_results, 
            style_map_country_year, 
            title=f"{sc} {score.upper()} {country_name} Market Energy Procurement Frontier"
        )
        title_fig = f"{score.upper()} Frontier - {sc} - {country_name}"
        
        if save_fig:
            country_fig_path = os.path.join(fig_path, country_name, sc)
            os.makedirs(country_fig_path, exist_ok=True)
            fig.savefig(os.path.join(country_fig_path, f"{title_fig}.png"), dpi=300, bbox_inches="tight")

        if not show_fig:
            plt.close()

## (a-3) Comparing different countries in the same year

In [ ]:
fig_path = "figures/energy_procurement_frontier/system"
style_map_country = {
    country: {'color': color, 'linewidth': 2, 'linestyle': 'solid'}
    for country, color in country_colors.items()
}

for sc in df_all.index:
    for year in df_all.columns:
        n = df_all.loc[sc, year]
        res, load = get_score_load(n, score=score)
        
        for i in [0,1,2]:
            data = {}
            
            for country in load.columns[i::3]:
                national_res = res[country]
                national_load = load[country]
                country_name = cc.convert(names=country, to='name_short')
                data[country_name] = get_hourly_energy_matrix(n, national_res, national_load)
        
            df_results = pd.DataFrame(data, index=range(1, 120))
            # df_results.to_csv(f"test_{scenario}_{score}.csv")
            fig = plot_hourly_energy_matrix(
                df_results, 
                style_map_country, 
                title=f"{sc} {score.upper()} National Market Energy Procurement Frontier part {i} in {year}"
            )
            title_fig = f"{score.upper()} Frontier - {sc} - Countries part {i} in {year}"
            
            if save_fig:
                eu_fig_path = os.path.join(fig_path, sc)
                os.makedirs(eu_fig_path, exist_ok=True)
                fig.savefig(os.path.join(eu_fig_path, f"{title_fig}.png"), dpi=300, bbox_inches="tight")

            if not show_fig:
                plt.close()